# Quantum Teleportation (Qiskit)

This notebook presents **Quantum Teleportation** with: 
- A clear introduction and intuition
- The full mathematical derivation
- A working Qiskit implementation
- High-precision circuit diagrams
- Verification (fidelity + sampling-based checks)


## 1. Introduction

Quantum teleportation is a protocol that transfers an **unknown quantum state** from Alice to Bob using:

- **One shared Bell pair** (pre-shared entanglement)
- **Two classical bits** sent from Alice to Bob
- **No physical transmission** of the message qubit itself

Teleportation does **not** violate relativity because Bob still needs Alice's **classical** message before he can finish the reconstruction.

The protocol also respects the **no-cloning theorem**: the original state is destroyed by Alice's measurement step.


## 2. Mathematical derivation (exact state evolution)

Let the unknown message qubit be

\[
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1.
\]

Alice and Bob share a Bell pair \(|\Phi^+\rangle\):

\[
|\Phi^+\rangle = \frac{1}{\sqrt{2}} (|00\rangle + |11\rangle).
\]

Label qubits as: 
- qubit 0: message (Alice)
- qubit 1: Alice's half of Bell pair
- qubit 2: Bob's half of Bell pair

Initial combined state:

\[
|\psi\rangle \otimes |\Phi^+\rangle \;=\; (\alpha|0\rangle + \beta|1\rangle) \otimes \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle).
\]

Alice applies the **Bell-basis measurement circuit** on qubits (0,1):
- CNOT from qubit 0 to qubit 1
- Hadamard on qubit 0

After these gates, the joint state can be rewritten as:

\[
\frac{1}{2} \Big( |00\rangle (\alpha|0\rangle + \beta|1\rangle) \; + \; |01\rangle (\alpha|1\rangle + \beta|0\rangle) \; + \; |10\rangle (\alpha|0\rangle - \beta|1\rangle) \; + \; |11\rangle (\alpha|1\rangle - \beta|0\rangle) \Big).
\]

Alice then measures qubits (0,1), obtaining two classical bits 
\(m_0, m_1\) \in \{0,1\}^2.

Bob's qubit collapses into one of four related states, and Bob applies corrections:

- If \((m_0, m_1) = (0,0)\): Bob already has \(|\psi\rangle\)
- If \((0,1)\): Bob applies \(X\)
- If \((1,0)\): Bob applies \(Z\)
- If \((1,1)\): Bob applies \(XZ\) (or Z then X, up to a global phase)

So in all cases, Bob ends with exactly the original state \(|\psi\rangle\).


In [4]:
import math
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity, random_statevector

try:
    from qiskit_aer import AerSimulator
    _AER_BACKEND = AerSimulator()
except Exception:
    try:
        from qiskit import Aer
        _AER_BACKEND = Aer.get_backend('aer_simulator')
    except Exception:
        _AER_BACKEND = None

plt.rcParams['figure.dpi'] = 140


In [ ]:
def make_message_state_on(qc: QuantumCircuit, q, theta: float, phi: float) -> None:
    """Prepare |psi> on qubit q using a Bloch-sphere parameterization.

    State prepared is: |psi> = RZ(phi) RY(theta) |0>
    where theta in [0, pi], phi in [0, 2pi).
    """
    qc.ry(theta, q)
    qc.rz(phi, q)


def apply_inverse_message_state_on(qc: QuantumCircuit, q, theta: float, phi: float) -> None:
    """Apply the inverse of make_message_state_on, so that it maps |psi> back to |0>."""
    qc.rz(-phi, q)
    qc.ry(-theta, q)


def _apply_classical_correction(qc: QuantumCircuit, gate_name: str, target, creg: ClassicalRegister, value: int) -> None:
    gate = getattr(qc, gate_name)

    if hasattr(qc, "if_test"):
        with qc.if_test((creg, value)):
            gate(target)
        return

    inst_set = gate(target)
    if hasattr(inst_set, "c_if"):
        inst_set.c_if(creg, value)
        return

    qc.data[-1].operation.condition = (creg, value)


def build_teleportation_circuit_measurement(theta: float, phi: float) -> QuantumCircuit:
    """Standard teleportation circuit: measurements + classically controlled corrections."""
    q = QuantumRegister(3, name='q')
    c0 = ClassicalRegister(1, name='m0')
    c1 = ClassicalRegister(1, name='m1')
    cb = ClassicalRegister(1, name='bob')
    qc = QuantumCircuit(q, c0, c1, cb, name='Teleport(meas)')

    make_message_state_on(qc, q[0], theta, phi)

    qc.h(q[1])
    qc.cx(q[1], q[2])

    qc.cx(q[0], q[1])
    qc.h(q[0])

    qc.measure(q[0], c0[0])
    qc.measure(q[1], c1[0])

    _apply_classical_correction(qc, "x", q[2], c1, 1)
    _apply_classical_correction(qc, "z", q[2], c0, 1)

    apply_inverse_message_state_on(qc, q[2], theta, phi)
    qc.measure(q[2], cb[0])

    return qc


def build_teleportation_circuit_deferred(theta: float, phi: float) -> QuantumCircuit:
    """Teleportation using deferred measurement principle (no intermediate measurements)."""
    q = QuantumRegister(3, name='q')
    cb = ClassicalRegister(1, name='bob')
    qc = QuantumCircuit(q, cb, name='Teleport(deferred)')

    make_message_state_on(qc, q[0], theta, phi)

    qc.h(q[1])
    qc.cx(q[1], q[2])

    qc.cx(q[0], q[1])
    qc.h(q[0])

    qc.cx(q[1], q[2])
    qc.cz(q[0], q[2])

    apply_inverse_message_state_on(qc, q[2], theta, phi)
    qc.measure(q[2], cb[0])

    return qc


def build_quantum_3_partition_teleportation_circuit(theta: float, phi: float) -> QuantumCircuit:
    q = QuantumRegister(3, name='q')
    cb = ClassicalRegister(1, name='bob')
    qc = QuantumCircuit(q, cb, name='Teleport(3-partition)')

    prep = QuantumCircuit(q, name='Preparation')
    make_message_state_on(prep, q[0], theta, phi)
    prep.h(q[1])
    prep.cx(q[1], q[2])

    alice = QuantumCircuit(q, name='Alice')
    alice.cx(q[0], q[1])
    alice.h(q[0])

    bob = QuantumCircuit(q, name='Bob')
    bob.cx(q[1], q[2])
    bob.cz(q[0], q[2])

    qc.compose(prep, inplace=True)
    qc.barrier(q)
    qc.compose(alice, inplace=True)
    qc.barrier(q)
    qc.compose(bob, inplace=True)
    qc.barrier(q)

    apply_inverse_message_state_on(qc, q[2], theta, phi)
    qc.measure(q[2], cb[0])

    return qc


## 3. Circuit diagrams (high precision)

We draw two circuits:

- **Measurement-based teleportation** (standard protocol)
- **Deferred-measurement teleportation** (fully unitary, coherent corrections)

Both should implement the same logical transfer of the unknown quantum state.


In [ ]:
theta = 0.7
phi = 1.2

qc_meas = build_teleportation_circuit_measurement(theta, phi)
qc_def = build_teleportation_circuit_deferred(theta, phi)

print('--- Measurement-based teleportation (TEXT) ---')
print(qc_meas.draw(output='text', fold=-1))

print('\n--- Deferred-measurement teleportation (TEXT) ---')
print(qc_def.draw(output='text', fold=-1))

try:
    fig1 = qc_meas.draw(output='mpl', fold=-1, idle_wires=False, scale=0.9)
    plt.show()
except Exception as e:
    print(f"\n(mpl draw failed for measurement-based circuit: {e})")

try:
    fig2 = qc_def.draw(output='mpl', fold=-1, idle_wires=False, scale=0.9)
    plt.show()
except Exception as e:
    print(f"\n(mpl draw failed for deferred-measurement circuit: {e})")


## 3B. Quantum 3D Partition circuit (Preparation / Alice / Bob)

This section draws a **3-partition (stage-separated)** teleportation circuit:

- **Partition 1 (Preparation):** prepares the unknown message state and the Bell pair
- **Partition 2 (Alice):** performs the Bell-basis entangling operations (CNOT + H)
- **Partition 3 (Bob):** applies coherent corrections (deferred-measurement principle)

The partitions are separated using `barrier` so the diagram is visually clean and “industrial-ready.”


In [ ]:
qc_3part = build_quantum_3_partition_teleportation_circuit(theta, phi)

print('--- Quantum 3-partition teleportation (TEXT) ---')
print(qc_3part.draw(output='text', fold=-1))

try:
    fig3 = qc_3part.draw(output='mpl', fold=-1, idle_wires=False, scale=0.9)
    plt.show()
except Exception as e:
    print(f"\n(mpl draw failed for 3-partition circuit: {e})")


## 4. Verification A (exact): state fidelity on Bob's qubit

To do a strict check, we simulate a **fully unitary** teleportation circuit (deferred-measurement form) and compute
the fidelity between the original message state and Bob's reduced state.

We compute fidelity: 
\[ F(|\psi\rangle, \rho_{Bob}) = \langle\psi| \rho_{Bob} |\psi\rangle. \]

For correct teleportation, this should be **1.0** (up to floating-point tolerance).


In [7]:
psi = random_statevector(2, seed=7)

q = QuantumRegister(3, name='q')
qc = QuantumCircuit(q)

qc.initialize(psi.data, q[0])
qc.h(q[1])
qc.cx(q[1], q[2])
qc.cx(q[0], q[1])
qc.h(q[0])
qc.cx(q[1], q[2])
qc.cz(q[0], q[2])

final = Statevector.from_instruction(qc)
rho = DensityMatrix(final)
rho_bob = partial_trace(rho, [0, 1])

F = state_fidelity(psi, rho_bob)
F


0.9999999999999998

## 5. Verification B (sampling): teleport and uncompute-to-|0⟩ test

For the measurement-based circuit, an easy verification is:

- Prepare a known state \(|\psi\rangle\)
- Teleport it
- Apply the inverse of the preparation unitary on Bob
- Measure Bob

If Bob really has \(|\psi\rangle\), the inverse will return him to \(|0\rangle\), so measurement should yield **0 with probability ~1**.


In [8]:
test_states = [
    ('|0>', 0.0, 0.0),
    ('|1>', math.pi, 0.0),
    ('|+>', math.pi / 2, 0.0),
    ('|->', math.pi / 2, math.pi),
    ('|+i>', math.pi / 2, math.pi / 2),
    ('|-i>', math.pi / 2, 3 * math.pi / 2),
]

if _AER_BACKEND is None:
    raise RuntimeError('Aer simulator backend not found. Install qiskit-aer to run shot-based verification.')

shots = 4096

rows = []
for label, th, ph in test_states:
    qc = build_teleportation_circuit_measurement(th, ph)
    tqc = transpile(qc, _AER_BACKEND)
    result = _AER_BACKEND.run(tqc, shots=shots).result()
    counts = result.get_counts()

    # The circuit has 3 classical bits total (m0, m1, bob). Qiskit returns bitstrings with
    # the highest-index classical bit on the left. Since bob is measured into the last-added
    # classical bit, bob is the leftmost bit after removing spaces.
    bob_counts = {'0': 0, '1': 0}
    for k, v in counts.items():
        bits = k.replace(' ', '')
        bob_bit = bits[0]
        bob_counts[bob_bit] += v

    p0 = bob_counts['0'] / shots
    p1 = bob_counts['1'] / shots
    rows.append((label, p0, p1, bob_counts))

rows


[('|0>', 1.0, 0.0, {'0': 4096, '1': 0}),
 ('|1>', 1.0, 0.0, {'0': 4096, '1': 0}),
 ('|+>', 1.0, 0.0, {'0': 4096, '1': 0}),
 ('|->', 1.0, 0.0, {'0': 4096, '1': 0}),
 ('|+i>', 1.0, 0.0, {'0': 4096, '1': 0}),
 ('|-i>', 1.0, 0.0, {'0': 4096, '1': 0})]